# 🧠 MedSSL-BHSD: PICK Semi-Supervised Segmentation trên Dual GPU T4 (Kaggle)

Phân đoạn 5 phân loại xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background trên 2x GPU T4 Kaggle theo kiến trúc MedSSL-BHSD tái cấu trúc.

### 🌟 Các cải tiến & Tối ưu hóa sẵn sàng cho 1 lần Run duy nhất:
- **Thí nghiệm Mới Độc lập**: `EXP_NAME` được đổi thành `BHSD_multiclass_PICK_dual_gpu_fold0` đảm bảo không bị trùng lặp với artifact cũ của notebook trước.
- **Cơ chế lưu Checkpoint Kép (Dual Checkpointing)**:
  - **Mỗi 50 iterations (`--save_interval 50`)**: Tự động lưu đè `checkpoint_latest.pth` cục bộ trên SSD tốc độ cao (chỉ mất ~0.2s), giúp chống crash/timeout mà không gây nghẽn mạng.
  - **Mỗi 500 iterations (`--val_interval 500`)**: Chạy 3D sliding-window validation 38 ca, cập nhật `VNet_best_model.pth` và đẩy đồng thời lên W&B Cloud Artifacts.
- **Chế độ 2 GPU T4 tối ưu (DDP - DistributedDataParallel qua `torchrun`)**: Tự động kích hoạt khi phát hiện 2 GPU, chia đều VRAM (~6GB mỗi GPU) và chạy nhanh hơn 25-30% so với DataParallel cũ.
- **Tự động đăng nhập W&B**: Khóa API Key đã được tích hợp sẵn, tự động đồng bộ biểu đồ mDice, per-class Dice và VRAM theo thời gian thực.
- **Khả năng Auto-Resume**: Tự động khôi phục từ checkpoint gần nhất nếu bị timeout 12h của Kaggle.
- **Đo lường Benchmark độc lập**: Tự động chạy `evaluate.py` sau huấn luyện để đo FLOPs, Latency, và per-class Dice trên sliding-window.
- **Tự động đóng gói kết quả**: Nén toàn bộ checkpoint tốt nhất và logs ra file `.tar.gz` sẵn sàng tải về tại tab Output.

### 📦 Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
%pip install -q medpy tensorboardX h5py SimpleITK nibabel wandb pyyaml

### 🚀 Bước 2: Clone & Cập nhật mã nguồn MedSSL-BHSD từ GitHub

In [ ]:
import os
import subprocess

REPO_NAME = "NguyenVanCongThanh/PICK_BHSD"
GITHUB_TOKEN = ""  # Điền token ghp_xxx nếu repo là Private, nếu Public thì để trống
WORK_DIR = "/kaggle/working/PICK"

if GITHUB_TOKEN.strip():
    GITHUB_REPO_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_NAME}.git"
else:
    GITHUB_REPO_URL = f"https://github.com/{REPO_NAME}.git"

if not os.path.exists(WORK_DIR):
    print(f"[*] Đang clone repository {REPO_NAME}...")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, WORK_DIR], check=True)
else:
    print(f"[OK] Cập nhật git pull tại {WORK_DIR}...")
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

%cd {WORK_DIR}
!git log -1 --oneline

### 🔍 Bước 3: Định vị Dataset BHSD từ Kaggle Input

In [ ]:
import os
import glob

INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

def find_bhsd_root(base_dir):
    if not os.path.exists(base_dir):
        return None
    for root, dirs, files in os.walk(base_dir):
        if "data" in dirs and ("splits" in dirs or "fold0_train.txt" in files):
            return root
    return None

print("[*] Đang định vị dataset BHSD...")
DATA_ROOT = find_bhsd_root(INPUT_DIR) or find_bhsd_root(WORKING_DIR)

if not DATA_ROOT:
    all_tars = glob.glob(os.path.join(INPUT_DIR, "**", "*.tar*"), recursive=True)
    matching_tars = [p for p in all_tars if any(k in os.path.basename(p).lower() for k in ["bhsd", "h5", "fold"])]
    tar_path = matching_tars[0] if matching_tars else (all_tars[0] if all_tars else None)
    
    if not tar_path:
        raise FileNotFoundError(f"[X] Không tìm thấy dataset BHSD trong {INPUT_DIR}! Hãy bấm '+ Add Input'.")
    
    print(f"[*] Đang giải nén {os.path.basename(tar_path)} vào {WORKING_DIR} (khoảng 1-2 phút)...")
    if tar_path.endswith(".tar.gz") or tar_path.endswith(".tgz"):
        !tar -xzf "{tar_path}" -C {WORKING_DIR}
    else:
        !tar -xf "{tar_path}" -C {WORKING_DIR}
    DATA_ROOT = find_bhsd_root(WORKING_DIR)
    assert DATA_ROOT, f"[X] Giải nén xong nhưng không tìm thấy data/ và splits/ trong {WORKING_DIR}!"

print(f"[SUCCESS] Đường dẫn DATA_ROOT: {DATA_ROOT}")

# Kiểm tra tính toàn vẹn dữ liệu
splits_dir = os.path.join(DATA_ROOT, "splits") if os.path.exists(os.path.join(DATA_ROOT, "splits")) else DATA_ROOT
data_dir = os.path.join(DATA_ROOT, "data")
h5_files = glob.glob(os.path.join(data_dir, "*.h5"))
print(f"  [+] Tổng số file .h5: {len(h5_files)}")
for f in range(5):
    f_train = os.path.join(splits_dir, f"fold{f}_train.txt")
    f_val = os.path.join(splits_dir, f"fold{f}_val.txt")
    if os.path.exists(f_train) and os.path.exists(f_val):
        n_tr = len([l for l in open(f_train) if l.strip()])
        n_va = len([l for l in open(f_val) if l.strip()])
        print(f"  [+] Fold {f}: {n_tr} train cases | {n_va} val cases")
print("[OK] Dữ liệu đã sẵn sàng!")

### ⚡ Bước 4: Kiểm tra Cấu hình Phần cứng Dual GPU (2x T4)

In [ ]:
import torch

print("=" * 60)
assert torch.cuda.is_available(), "[CẢNH BÁO] Chưa bật GPU! Hãy chọn GPU T4 x 2 ở mục Settings."
num_gpus = torch.cuda.device_count()
print(f"[GPU] Phát hiện {num_gpus} GPU khả dụng:")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)
    print(f"  [+] GPU {i}: {name} | VRAM: {vram} GB")

if num_gpus >= 2:
    print("[*] Đã sẵn sàng chế độ ĐA GPU (2x T4 - DDP via torchrun) để tăng tốc và cân bằng VRAM!")
else:
    print("[*] Chế độ đơn GPU (1x T4 hoặc P100) sẵn sàng.")
print("=" * 60)

### 📊 Bước 5: Đăng nhập & Cấu hình Weights & Biases (W&B)
API Key đã được tích hợp sẵn để đảm bảo notebook chạy thông suốt trong 1 lần run duy nhất.

In [ ]:
import os
import wandb

# API Key W&B tích hợp sẵn cho 1 lần Run
WANDB_API_KEY = "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"
WANDB_PROJECT = "PICK_BHSD"

# Thiết lập biến môi trường để cả tiến trình con (torchrun / python) tự động nhận diện
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

print("[*] Đang kết nối tới Weights & Biases...")
wandb.login(key=WANDB_API_KEY, relogin=True)
USE_WANDB = "true"
print(f"[SUCCESS] Đăng nhập W&B thành công! Dữ liệu huấn luyện sẽ được lưu tại project: '{WANDB_PROJECT}'")

### 🏋️‍♂️ Bước 6: Huấn luyện PICK Multi-Class với Kiến trúc Mới (2x T4 DDP)

Sử dụng entrypoint mới **`train.py`** cùng cấu hình **`configs/methods/pick.yaml`**:
- **Tên thí nghiệm mới**: `BHSD_multiclass_PICK_dual_gpu_fold0` (hoàn toàn mới, không ghi đè artifact cũ).
- **Lưu đè cục bộ mỗi 50 iterations (`--save_interval 50`)**: Chống crash và timeout tức thời mà không tốn băng thông Cloud.
- **Chế độ DDP (`torchrun --nproc_per_node=2`)**: Phân tán 2 process trên 2 GPU T4, chia đều VRAM (~6GB mỗi card) và đẩy tốc độ tối đa.
- **Auto-Resume**: Cờ `--resume` tự động tìm checkpoint gần nhất (`checkpoint_latest.pth`) để tiếp tục nếu gặp timeout.

In [ ]:
# === HUẤN LUYỆN PICK TRÊN 2 GPU T4 VỚI FRAMEWORK MỚI ===
import os
import sys
import subprocess
import torch

# --- THÔNG SỐ THÍ NGHIỆM ---
FOLD_ID = 0                                            # Chọn fold (0..4)
TASK = "multiclass"                                    # "multiclass" (6 lớp) hoặc "binary" (2 lớp)
MODEL_NAME = "vnet"                                    # "vnet", "unet3d", "unetr"
EXP_NAME = f"BHSD_{TASK}_PICK_dual_gpu_fold{FOLD_ID}"  # Tên thí nghiệm MỚI ĐỘC LẬP (tránh trùng artifact cũ)
EXP_DIR = "/kaggle/working/experiments"
CONFIG_FILE = "configs/methods/pick.yaml"
VAL_INTERVAL = 500                                     # Đánh giá validation và đẩy Cloud mỗi 500 bước
SAVE_INTERVAL = 50                                     # Tự động lưu checkpoint_latest.pth cục bộ mỗi 50 bước để chống crash
BATCH_SIZE = 4                                         # Batch size tổng (DDP: mỗi GPU xử lý 2 mẫu)
LABELED_BS = 2                                         # Số mẫu có nhãn trong batch

# Tự động phát hiện chế độ Đa GPU: Ưu tiên DDP nếu có >= 2 GPU
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_ddp = (num_gpus >= 2)

base_args = [
    "--config", CONFIG_FILE,
    "--fold", str(FOLD_ID),
    "--task", TASK,
    "--model", MODEL_NAME,
    "--exp", EXP_NAME,
    "--exp_dir", EXP_DIR,
    "--resume",
    "--root_path", str(DATA_ROOT),
    "--batch_size", str(BATCH_SIZE),
    "--labeled_bs", str(LABELED_BS),
    "--val_interval", str(VAL_INTERVAL),
    "--save_interval", str(SAVE_INTERVAL),
    "--use_wandb",
    "--save_wandb_model",
    "--wandb_project", str(WANDB_PROJECT)
]

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_API_KEY"] = WANDB_API_KEY

if use_ddp:
    print(f"[*] Phát hiện {num_gpus} GPU -> Khởi chạy chế độ DDP (torchrun) tối ưu trên 2x GPU T4...")
    train_cmd = [
        "torchrun",
        f"--nproc_per_node={num_gpus}",
        "--master_port=29500",
        "train.py",
        "--ddp"
    ] + base_args
else:
    print("[*] Chế độ: Đơn GPU hoặc DataParallel (DP) qua python...")
    train_cmd = [
        "python", "-u", "train.py",
        "--gpu", "auto"
    ] + base_args

print("=" * 75)
print(f"[*] Khởi chạy lệnh:\n{' '.join(train_cmd)}")
print(f"[*] Theo dõi biểu đồ W&B Cloud tại: https://wandb.ai/{WANDB_PROJECT}")
print("=" * 75)

# Stream log thời gian thực ra màn hình
process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

process.wait()

print("\n" + "=" * 75)
if process.returncode != 0:
    raise RuntimeError(f"[LỖI] Huấn luyện kết thúc với mã lỗi {process.returncode}!")
else:
    print(f"[SUCCESS] Huấn luyện hoàn tất thành công cho {EXP_NAME}!")
print("=" * 75)

### 🔬 Bước 7: Đánh giá Độc lập với Unified Benchmark (`evaluate.py`)
Nạp checkpoint tốt nhất (`VNet_best_model.pth`) để tính toán điểm mDice, per-class Dice và đo lường FLOPs, Latency.

In [ ]:
import os
import subprocess

snapshot_dir = os.path.join(EXP_DIR, EXP_NAME, f"fold_{FOLD_ID}")
best_pth = os.path.join(snapshot_dir, "self_train", "VNet_best_model.pth")
if not os.path.exists(best_pth):
    best_pth = os.path.join(snapshot_dir, "pre_train", "VNet_best_model.pth")

if os.path.exists(best_pth):
    print(f"[✓] Tìm thấy best model checkpoint: {best_pth}")
    eval_cmd = [
        "python", "evaluate.py",
        "--checkpoint", best_pth,
        "--root_path", str(DATA_ROOT),
        "--task", TASK,
        "--model", MODEL_NAME,
        "--fold", str(FOLD_ID),
        "--stride_xy", "32",
        "--stride_z", "8",
        "--profile"
    ]
    print(f"[*] Chạy đánh giá:\n{' '.join(eval_cmd)}\n")
    subprocess.run(eval_cmd, check=True)
else:
    print(f"[!] Chưa tìm thấy checkpoint tại {best_pth}. Hãy kiểm tra lại thư mục snapshot.")

### 📦 Bước 8: Đóng gói Kết quả Checkpoint & Logs
Nén toàn bộ kết quả mô hình thành file `.tar.gz` để tải về trực tiếp từ mục **Output** của Kaggle.

In [ ]:
import os

OUTPUT_ARCHIVE = f"/kaggle/working/{EXP_NAME}_results.tar.gz"

if os.path.exists(EXP_DIR):
    print(f"[*] Đang nén thư mục kết quả {EXP_DIR}...")
    !tar -czf {OUTPUT_ARCHIVE} -C {EXP_DIR} .
    size_mb = round(os.path.getsize(OUTPUT_ARCHIVE) / (1024**2), 2)
    print(f"\n[✓] Đã tạo thành công file lưu trữ: {OUTPUT_ARCHIVE} ({size_mb} MB)")
    print("[*] File đã sẵn sàng tải về tại mục Output bên phải màn hình.")
    print(f"[*] Xem lại toàn bộ biểu đồ và so sánh: https://wandb.ai/{WANDB_PROJECT}")
else:
    print(f"[!] Thư mục {EXP_DIR} không tồn tại.")